[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/05_pandas-grouping.ipynb)

*Python for Data Science · Notebook 5 of 8*

# Grouping, Merging & Reshaping

Split-apply-combine with groupby, joining tables with merge and concat, pivot tables, melt and time-based windows.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Groupby is split, apply, combine: sort the receipts into piles, total each pile, then write a summary.

**What interviewers really test here:** groupby vs transform, merges and their row-count surprises, pivots and time windows.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Split, apply, combine *(Basics)*
- [ ] 2. groupby by hand *(Basics)*
- [ ] 3. agg, transform and filter *(Core)*
- [ ] 4. Joining tables: merge and concat *(Core)*
- [ ] 5. agg vs transform: same size or smaller? *(Core)*
- [ ] 6. Joins by hand *(Core)*
- [ ] 7. Reshaping: pivot, melt, crosstab, stack *(Advanced)*
- [ ] 8. Time-based grouping and rolling windows *(Advanced)*
- [ ] 9. Wide and long, by hand *(Advanced)*
- [ ] 10. Rolling windows and changes over time, by hand *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Split-apply-combine

The **split-apply-combine** strategy (Hadley Wickham, 2011) underlies `groupby`: split the data into groups by key, apply a function independently to each group, and combine the results. It's the pandas equivalent of SQL's `GROUP BY`.



`df.groupby(keys)` returns a lazy **GroupBy** object; nothing is computed until you call an aggregation, transformation or filter. Keys can be column names, lists of them, Series, functions of the index, or `pd.Grouper(freq="M")` for time.



Options: `as_index=False` keeps keys as columns, `sort=False` keeps first-seen order, `dropna=False` keeps a NaN group, `observed=True` for categoricals.

### Aggregation vs transformation vs filtration

| Kind | Output shape | Examples |
| --- | --- | --- |
| **Aggregation** | one value per group | `sum mean median min max count size nunique std first last agg` |
| **Transformation** | same index as input | `transform("mean")`, `cumsum`, `rank`, `shift`, `fillna` within group |
| **Filtration** | subset of original rows | `filter(lambda g: …)`, `head(n)`, `nth` |



`count` excludes NaN; `size` counts all rows.

### Joins

| pandas | SQL | Rows kept |
| --- | --- | --- |
| `merge(how="inner")` | INNER JOIN | matching keys only |
| `merge(how="left")` | LEFT JOIN | all left rows |
| `merge(how="right")` | RIGHT JOIN | all right rows |
| `merge(how="outer")` | FULL OUTER JOIN | all keys |
| `merge(how="cross")` | CROSS JOIN | every pair |



Cardinality: one-to-one, one-to-many, many-to-many. Many-to-many multiplies rows. `validate="one_to_many"` makes pandas check the relationship; `indicator=True` adds a column saying where each row came from (useful for anti-joins). `join` merges on the index; `concat` stacks along an axis without matching keys.

### Tidy data and reshaping

**Tidy data**: each variable is a column, each observation is a row, each type of observational unit is a table. Long/tidy format is easiest to filter, group and plot.



| Function | Direction | Notes |
| --- | --- | --- |
| `pivot` | long → wide | no aggregation; fails on duplicate pairs |
| `pivot_table` | long → wide | aggregates duplicates (aggfunc), margins=True adds totals |
| `melt` | wide → long | id\_vars stay, other columns become variable/value |
| `stack / unstack` | columns ↔ index levels | work with MultiIndex |
| `crosstab` | frequency table | normalize= for proportions |

### MultiIndex

Grouping by several keys produces a **MultiIndex** — a hierarchical index with several levels (region, product). Access with tuples `s.loc[("North", "pen")]` or `xs`; flatten with `reset_index()`; move a level to columns with `unstack()`.

### Windows and time series

| Tool | Meaning |
| --- | --- |
| `rolling(n)` | fixed-size moving window (last n rows or a time span like "7D") |
| `expanding()` | everything up to the current row (running statistics) |
| `ewm(span=n)` | exponentially weighted (recent rows count more) |
| `resample("ME")` | group a DatetimeIndex by calendar period |
| `shift(k)`, `diff()`, `pct_change()` | lags, differences, growth rates |

---
# Part 2: Lessons, step by step

## Lesson 1: Split, apply, combine

*Level: Basics*

Most business questions sound like "**total sales per region**" or "**average score per class**". `groupby` answers them in three steps:




1. **Split** the rows into groups by a key (region).
2. **Apply** a function to each group (sum the amounts).
3. **Combine** the results into a new Series or DataFrame indexed by the key.



```python
df.groupby("region")["amount"].sum()
df.groupby("region")["amount"].mean()
df.groupby("region").size()               # rows per group
df.groupby(["region", "product"])["amount"].sum()   # two keys
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def groupby_demo():
    df = pd.DataFrame({"region": ["North", "South", "North", "South"], "amount": [100, 80, 40, 120]})
    totals = df.groupby("region")["amount"].sum()
    average = df.groupby("region")["amount"].mean()
    counts = df.groupby("region").size()
    print(totals.to_dict(), average.to_dict(), counts.to_dict())


groupby_demo()

{'North': 140, 'South': 200} {'North': 70.0, 'South': 100.0} {'North': 2, 'South': 2}


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 2: groupby by hand

*Level: Basics*

Sales: Pune 10, Delhi 20, Pune 30, Delhi 40, Goa 50. `df.groupby("city")["sales"].sum()` runs three steps:



<table><tr><th>Split</th><th>Apply (sum)</th><th>Combine</th></tr> <tr><td>Delhi: 20, 40</td><td>60</td><td rowspan="3">Delhi 60<br>Goa 50<br>Pune 40</td></tr> <tr><td>Goa: 50</td><td>50</td></tr> <tr><td>Pune: 10, 30</td><td>40</td></tr></table>



The result is sorted by group name, and the groups become the index. `as_index=False` or `.reset_index()` gives a normal table.



#### Common points



- Rows with a missing group value are **dropped** unless you use `dropna=False`. Totals can come out short without warning.
- `size()` counts rows, while `count()` counts non-missing values.
- Name your results: `agg(total=("sales", "sum"), average=("sales", "mean"))`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def groupby_by_hand():
    df = pd.DataFrame({"city": ["Pune", "Delhi", "Pune", "Delhi", "Goa", None], "sales": [10, 20, 30, 40, 50, 60]})
    totals = df.groupby("city")["sales"].sum()
    with_missing_group = df.groupby("city", dropna=False)["sales"].sum()
    summary = df.groupby("city").agg(total=("sales", "sum"), average=("sales", "mean"), orders=("sales", "size"))
    as_table = totals.reset_index()


groupby_by_hand()

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: agg, transform and filter

*Level: Core*

| Method | Returns | Use for |
| --- | --- | --- |
| `agg` | one row per group | summaries: `agg(total=("amount", "sum"), avg=("amount", "mean"))` |
| `transform` | same length as the original | group values per row: share of group total, z-score within group |
| `filter` | original rows of some groups | keep groups that pass a test (≥ 3 sales) |
| `apply` | anything | custom logic (slowest) |



Also useful per group: `cumsum`, `rank`, `first`/`last`, `nlargest`, `idxmax`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def agg_transform_demo():
    df = pd.DataFrame({"region": ["N", "S", "N", "S", "N"], "amount": [100, 80, 40, 120, 60]})
    summary = df.groupby("region").agg(total=("amount", "sum"), biggest=("amount", "max"))
    df["region_total"] = df.groupby("region")["amount"].transform("sum")
    df["share"] = (df["amount"] / df["region_total"] * 100).round(1)
    busy = df.groupby("region").filter(lambda g: len(g) >= 3)
    print(summary)
    print(df)
    print(busy["amount"].tolist())


agg_transform_demo()

        total  biggest
region                
N         200      100
S         200      120
  region  amount  region_total  share
0      N     100           200   50.0
1      S      80           200   40.0
2      N      40           200   20.0
3      S     120           200   60.0
4      N      60           200   30.0
[100, 40, 60]


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q13](#q13), [Q19](#q19), [Q14](#q14), [Q15](#q15), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 4: Joining tables: merge and concat

*Level: Core*

`pd.merge` combines tables on matching key columns, like SQL JOIN:



| `how=` | Keeps |
| --- | --- |
| `"inner"` (default) | only keys in both tables |
| `"left"` | every row of the left table (missing matches → NaN) |
| `"right"` | every row of the right table |
| `"outer"` | every key from either table |



```python
pd.merge(orders, customers, left_on="cust", right_on="id", how="left")
pd.concat([jan, feb], ignore_index=True)       # stack rows
pd.concat([a, b], axis=1)                      # side by side (aligned on index)
```



**⚠️ Watch out:** If keys repeat on both sides, merge produces every combination — row counts can explode. Check `len()` before and after.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def merge_demo():
    customers = pd.DataFrame({"id": [1, 2, 3], "name": ["Ana", "Ben", "Cy"]})
    orders = pd.DataFrame({"cust": [2, 1, 2, 9], "amount": [50, 30, 20, 99]})
    inner = orders.merge(customers, left_on="cust", right_on="id")
    left = customers.merge(orders, left_on="id", right_on="cust", how="left")
    print(len(inner), len(left))
    print(left[["name", "amount"]])


merge_demo()

3 4
  name  amount
0  Ana    30.0
1  Ben    50.0
2  Ben    20.0
3   Cy     NaN


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q9](#q9), [Q4](#q4), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 5: agg vs transform: same size or smaller?

*Level: Core*

|   | Output size | Use it to |
| --- | --- | --- |
| `agg` | one row per group | summarise |
| `transform` | same rows as the input | add a group result to every row |
| `filter` | some of the original rows | keep or drop whole groups |



#### Worked example: share of region total



| Region | Sales | transform("sum") | Share |
| --- | --- | --- | --- |
| A | 10 | 40 | 0.25 |
| A | 30 | 40 | 0.75 |
| B | 5 | 20 | 0.25 |
| B | 15 | 20 | 0.75 |



```python
df["share"] = df["sales"] / df.groupby("region")["sales"].transform("sum")
```



Because `transform` returns one value per original row, the division lines up. `sum()` alone returns just 2 rows, which wouldn't.



The same idea gives rank within a group, difference from a group average, and running totals with `cumsum`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def transform_vs_agg():
    df = pd.DataFrame({"region": ["A", "A", "B", "B"], "sales": [10, 30, 5, 15]})
    summary = df.groupby("region")["sales"].sum()
    same_size = df.groupby("region")["sales"].transform("sum")
    df["share"] = df["sales"] / same_size
    df["rank_in_region"] = df.groupby("region")["sales"].rank(ascending=False)
    df["running"] = df.groupby("region")["sales"].cumsum()


transform_vs_agg()

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q19](#q19), [Q7](#q7), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 6: Joins by hand

*Level: Core*

Customers: 1 Ana, 2 Ben, 3 Cy. Orders by customer id: 1, 1, 2, 4.



| how= | Keeps | Rows | Result |
| --- | --- | --- | --- |
| inner | matches only | 3 | Ana, Ana, Ben |
| left | every customer | 4 | Ana, Ana, Ben, Cy (no order) |
| right | every order | 4 | Ana, Ana, Ben, and an order for unknown customer 4 |
| outer | everything | 5 | all of the above |



#### The row explosion



If a key repeats on **both** sides, rows multiply: 2 rows × 3 rows with the same key give 6. This silently inflates totals. Check the row count after every join, and let pandas check it for you:



```python
orders.merge(customers, on="id", validate="many_to_one", indicator=True)
```



`validate` raises an error if the keys aren't as expected, and `indicator=True` adds a column saying where each row came from. To find customers without orders, keep the rows marked `left_only`.



**Check key types too:** an id stored as text won't match one stored as a number.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def joins_by_hand():
    customers = pd.DataFrame({"id": [1, 2, 3], "name": ["Ana", "Ben", "Cy"]})
    orders = pd.DataFrame({"id": [1, 1, 2, 4], "amount": [10, 20, 30, 40]})
    sizes = {how: len(customers.merge(orders, on="id", how=how)) for how in ["inner", "left", "right", "outer"]}
    marked = customers.merge(orders, on="id", how="left", indicator=True)
    no_orders = marked[marked["_merge"] == "left_only"]["name"]
    a = pd.DataFrame({"k": [1, 1], "x": [1, 2]})
    b = pd.DataFrame({"k": [1, 1, 1], "y": [7, 8, 9]})
    exploded = len(a.merge(b, on="k"))


joins_by_hand()

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q9](#q9), [Q20](#q20), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 7: Reshaping: pivot, melt, crosstab, stack

*Level: Advanced*

**Wide** data has one column per category (Jan, Feb, Mar). **Long** (tidy) data has one row per observation (region, month, amount). Analysis and Seaborn prefer long; reports prefer wide.



| Long → wide | Wide → long |
| --- | --- |
| `pivot_table(index, columns, values, aggfunc)` | `melt(id_vars, var_name, value_name)` |
| `unstack()` (index level → columns) | `stack()` (columns → index level) |
| `pd.crosstab(a, b)` (counts) | — |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def reshape_demo():
    sales = pd.DataFrame({"region": ["N", "S", "N", "S"], "month": ["Jan", "Jan", "Feb", "Feb"], "amount": [100, 80, 40, 120]})
    wide = sales.pivot_table(index="region", columns="month", values="amount", aggfunc="sum")
    long = wide.reset_index().melt(id_vars="region", value_name="amount")
    counts = pd.crosstab(sales["region"], sales["month"])
    print(wide)
    print(long)
    print(counts)


reshape_demo()

month   Feb  Jan
region          
N        40  100
S       120   80
  region month  amount
0      N   Feb      40
1      S   Feb     120
2      N   Jan     100
3      S   Jan      80
month   Feb  Jan
region          
N         1    1
S         1    1


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 8: Time-based grouping and rolling windows

*Level: Advanced*

```python
df["date"] = pd.to_datetime(df["date"])
df.groupby(df["date"].dt.to_period("M"))["amount"].sum()      # per month
df.resample("ME", on="date")["amount"].sum()                  # same, with resample
df["amount"].rolling(7).mean()                                # 7-row moving average
df["amount"].pct_change()                                     # change vs previous row
df["amount"].shift(1)                                         # previous row's value
```



Rolling windows smooth noisy daily data; the first `window − 1` results are NaN because there isn't enough history yet.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [9]:
def time_demo():
    df = pd.DataFrame({"date": pd.date_range("2024-01-30", periods=6, freq="D"), "amount": [10, 20, 30, 40, 50, 60]})
    monthly = df.groupby(df["date"].dt.to_period("M"))["amount"].sum()
    df["avg3"] = df["amount"].rolling(3).mean()
    df["change"] = df["amount"].pct_change().round(2)
    print(monthly.to_dict())
    print(df)


time_demo()

{Period('2024-01', 'M'): 30, Period('2024-02', 'M'): 180}
        date  amount  avg3  change
0 2024-01-30      10   NaN     NaN
1 2024-01-31      20   NaN    1.00
2 2024-02-01      30  20.0    0.50
3 2024-02-02      40  30.0    0.33
4 2024-02-03      50  40.0    0.25
5 2024-02-04      60  50.0    0.20


**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 9: Wide and long, by hand

*Level: Advanced*

| Name | Jan | Feb |
| --- | --- | --- |
| Ana | 10 | 20 |
| Ben | 30 | 40 |



That's **wide**: good for reading. Plotting and `groupby` prefer **long**, with one observation per row:



| Name | Month | Sales |
| --- | --- | --- |
| Ana | Jan | 10 |
| Ben | Jan | 30 |
| Ana | Feb | 20 |
| Ben | Feb | 40 |



```python
long = wide.melt(id_vars="name", var_name="month", value_name="sales")
wide = long.pivot(index="name", columns="month", values="sales")
```



#### pivot or pivot\_table?



- `pivot` only reshapes. Repeated name–month pairs raise an error.
- `pivot_table` also combines repeats, with the mean by default. So pass `aggfunc="sum"` if totals are what you want.



**Cross-tab with shares:** `pd.crosstab(a, b, normalize="index")` shows the share of each row.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [10]:
def reshape_by_hand():
    wide = pd.DataFrame({"name": ["Ana", "Ben"], "Jan": [10, 30], "Feb": [20, 40]})
    long = wide.melt(id_vars="name", var_name="month", value_name="sales")
    back = long.pivot(index="name", columns="month", values="sales")
    repeats = pd.DataFrame({"name": ["Ana", "Ana"], "month": ["Jan", "Jan"], "sales": [10, 5]})
    default_is_mean = repeats.pivot_table(index="name", columns="month", values="sales")
    summed = repeats.pivot_table(index="name", columns="month", values="sales", aggfunc="sum")


reshape_by_hand()

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q10](#q10), [Q12](#q12), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 10: Rolling windows and changes over time, by hand

*Level: Advanced*

Values: 10, 20, 30, 40, 50.



| Value | rolling(3).mean() | shift(1) | diff() | pct\_change() | cumsum() |
| --- | --- | --- | --- | --- | --- |
| 10 | – | – | – | – | 10 |
| 20 | – | 10 | 10 | 1.00 | 30 |
| 30 | 20 | 20 | 10 | 0.50 | 60 |
| 40 | 30 | 30 | 10 | 0.33 | 100 |
| 50 | 40 | 40 | 10 | 0.25 | 150 |



The first two rolling values are blank because a window of 3 isn't full yet (`min_periods=1` changes that). Rolling windows look **backwards** by default, so the value on each row uses that row and earlier ones.



#### Time pitfalls



- Sort by date first; these all use row order.
- `rolling("7D")` uses real time and copes with missing days, while `rolling(7)` just counts 7 rows.
- In ML features, use `shift(1)` before rolling, or today's value leaks into the prediction of today.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [11]:
def rolling_by_hand():
    s = pd.Series([10, 20, 30, 40, 50])
    rolling = s.rolling(3).mean()
    previous = s.shift(1)
    change = s.diff()
    percent = s.pct_change().round(2)
    running = s.cumsum()
    safe_feature = s.shift(1).rolling(3).mean()


rolling_by_hand()

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q16](#q16). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Total per Group

🟢 Warm-up · *groupby + sum*

Return the total amount for each region, as {region: total} sorted by region name.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'East': 60, 'North': 210, 'South': 200}`

In [ ]:
def region_totals(data):
    """
    Return the total amount for each region, as {region: total} sorted by region
    name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(region_totals, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'East': 60, 'North': 210, 'South': 200}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Accumulate totals in a dict, then sort the keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('region')['amount'].sum()` — split, apply sum, combine.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Accumulate totals in a dict, then sort the keys.

```python
def region_totals(data):
    totals = {}
    for r, a in zip(data["region"], data["amount"]):
        totals[r] = totals.get(r, 0) + a
    return dict(sorted(totals.items()))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('region')['amount'].sum()` — split, apply sum, combine.

```python
def region_totals(data):
    df = pd.DataFrame(data)
    return df.groupby("region")["amount"].sum()
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Accumulate totals in a dict, then sort the keys. That costs O(n) time and O(k) space."

"I can do better: `df.groupby('region')['amount'].sum()` — split, apply sum, combine. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/total-per-group)

<a id="q2"></a>
### Q2. Average per Group

🟢 Warm-up · *groupby + mean*

Return the average amount per product, rounded to 2 decimals.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'ink': 80.0, 'pen': 77.5}`

In [ ]:
def product_avg(data):
    """
    Return the average amount per product, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(product_avg, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'ink': 80.0, 'pen': 77.5}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a sum and a count per product.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('product')['amount'].mean()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep a sum and a count per product.

```python
def product_avg(data):
    sums, counts = {}, {}
    for p, a in zip(data["product"], data["amount"]):
        sums[p] = sums.get(p, 0) + a
        counts[p] = counts.get(p, 0) + 1
    return {p: round(sums[p] / counts[p], 2) for p in sorted(sums)}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('product')['amount'].mean()`.

```python
def product_avg(data):
    return pd.DataFrame(data).groupby("product")["amount"].mean().round(2)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a sum and a count per product. That costs O(n) time and O(k) space."

"I can do better: `df.groupby('product')['amount'].mean()`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/average-per-group)

<a id="q3"></a>
### Q3. Rows per Group

🟢 Warm-up · *groupby + size*

Return how many sales each region has.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'East': 1, 'North': 3, 'South': 2}`

In [ ]:
def rows_per_region(data):
    """
    Return how many sales each region has.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rows_per_region, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'East': 1, 'North': 3, 'South': 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('region').size()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count in a dict.

```python
def rows_per_region(data):
    counts = {}
    for r in data["region"]:
        counts[r] = counts.get(r, 0) + 1
    return dict(sorted(counts.items()))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('region').size()`.

```python
def rows_per_region(data):
    return pd.DataFrame(data).groupby("region").size()
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in a dict. That costs O(n) time and O(k) space."

"I can do better: `df.groupby('region').size()`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/rows-per-group)

<a id="q4"></a>
### Q4. Stack Two Tables

🟢 Warm-up · *concat*

Combine two tables with the same columns, one under the other, and return the names.

**Examples**

- `a = {'name': ['Ana', 'Ben']}, b = {'name': ['Cy']}` → `['Ana', 'Ben', 'Cy']`

In [ ]:
def stack_tables(a, b):
    """
    Combine two tables with the same columns, one under the other, and return
    the names.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stack_tables, [
    (({'name': ['Ana', 'Ben']}, {'name': ['Cy']}), ['Ana', 'Ben', 'Cy']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

List concatenation of each column.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.concat([a, b], ignore_index=True)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** List concatenation of each column.

```python
def stack_tables(a, b):
    return list(a["name"]) + list(b["name"])
```

**Cost:** O(n + m) time, O(n + m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.concat([a, b], ignore_index=True)`.

```python
def stack_tables(a, b):
    both = pd.concat([pd.DataFrame(a), pd.DataFrame(b)], ignore_index=True)
    return both["name"].tolist()
```

**Cost:** O(n + m) time, O(n + m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: List concatenation of each column. That costs O(n + m) time and O(n + m) space."

"I can do better: `pd.concat([a, b], ignore_index=True)`. That's O(n + m) time and O(n + m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/stack-two-tables)

<a id="q5"></a>
### Q5. First and Last per Group

🟢 Warm-up · *groupby + first / last*

For each region, return (first amount, last amount) in row order.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'East': (60, 60), 'North': (100, 70), 'South': (80, 120)}`

In [ ]:
def first_last(data):
    """
    For each region, return (first amount, last amount) in row order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_last, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'East': (60, 60), 'North': (100, 70), 'South': (80, 120)}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the first and latest value per region in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groupby('region')['amount'].agg(['first', 'last'])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Track the first and latest value per region in a dict.

```python
def first_last(data):
    out = {}
    for r, a in zip(data["region"], data["amount"]):
        if r not in out:
            out[r] = [a, a]
        out[r][1] = a
    return {r: tuple(out[r]) for r in sorted(out)}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `groupby('region')['amount'].agg(['first', 'last'])`.

```python
def first_last(data):
    fl = pd.DataFrame(data).groupby("region")["amount"].agg(["first", "last"])
    return {r: (row["first"], row["last"]) for r, row in fl.iterrows()}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the first and latest value per region in a dict. That costs O(n) time and O(k) space."

"I can do better: `groupby('region')['amount'].agg(['first', 'last'])`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/first-and-last-per-group)

<a id="q6"></a>
### Q6. Several Statistics at Once

🟡 Interview standard · *agg with named outputs*

Per region, return total, average (2 decimals) and largest amount as {'total': {...}, 'avg': {...}, 'max': {...}}.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'total': [60, 210, 200], 'avg': [60.0, 70.0, 100.0], 'max': [60, 100, 120]}`

In [ ]:
def region_stats(data):
    """
    Per region, return total, average (2 decimals) and largest amount as
    {'total': {...}, 'avg': {...}, 'max': {...}}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(region_stats, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'total': [60, 210, 200], 'avg': [60.0, 70.0, 100.0], 'max': [60, 100, 120]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group values into lists per region, then compute each statistic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Named aggregation: `agg(total=('amount', 'sum'), …)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Group values into lists per region, then compute each statistic.

```python
def region_stats(data):
    groups = {}
    for r, a in zip(data["region"], data["amount"]):
        groups.setdefault(r, []).append(a)
    regions = sorted(groups)
    return {"total": [sum(groups[r]) for r in regions],
            "avg": [round(sum(groups[r]) / len(groups[r]), 2) for r in regions],
            "max": [max(groups[r]) for r in regions]}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Named aggregation: `agg(total=('amount', 'sum'), …)`.

```python
def region_stats(data):
    df = pd.DataFrame(data)
    stats = df.groupby("region").agg(total=("amount", "sum"), avg=("amount", "mean"), max=("amount", "max"))
    return stats.round(2)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group values into lists per region, then compute each statistic. That costs O(n) time and O(n) space."

"I can do better: Named aggregation: `agg(total=('amount', 'sum'), …)`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/several-statistics-at-once)

<a id="q7"></a>
### Q7. Best Product per Region

🟡 Interview standard · *idxmax within groups*

For each region, return the product with the highest total amount.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'East': 'pen', 'North': 'pen', 'South': 'ink'}`

In [ ]:
def best_product(data):
    """
    For each region, return the product with the highest total amount.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(best_product, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'East': 'pen', 'North': 'pen', 'South': 'ink'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dicts of totals, then pick the max per region.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group by both, sum, then `idxmax` per region.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested dicts of totals, then pick the max per region.

```python
def best_product(data):
    totals = {}
    for r, p, a in zip(data["region"], data["product"], data["amount"]):
        totals.setdefault(r, {})
        totals[r][p] = totals[r].get(p, 0) + a
    return {r: max(totals[r], key=totals[r].get) for r in sorted(totals)}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Group by both, sum, then `idxmax` per region.

```python
def best_product(data):
    df = pd.DataFrame(data)
    totals = df.groupby(["region", "product"])["amount"].sum().reset_index()
    best = totals.loc[totals.groupby("region")["amount"].idxmax()]
    return dict(zip(best["region"], best["product"]))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dicts of totals, then pick the max per region. That costs O(n) time and O(n) space."

"I can do better: Group by both, sum, then `idxmax` per region. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/best-product-per-region)

<a id="q8"></a>
### Q8. Join Customers and Orders

🟡 Interview standard · *merge (inner)*

Return (customer name, order amount) for every order whose customer id exists.

**Examples**

- `customers = {'id': [1, 2, 3], 'name': ['Ana', 'Ben', 'Cy']}, orders = {'cust': [2, 1, 2, 9], 'amount': [50, 30, 20, 99]}` → `[('Ben', 50), ('Ana', 30), ('Ben', 20)]`

In [ ]:
def join_orders(customers, orders):
    """
    Return (customer name, order amount) for every order whose customer id
    exists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(join_orders, [
    (({'id': [1, 2, 3], 'name': ['Ana', 'Ben', 'Cy']}, {'cust': [2, 1, 2, 9], 'amount': [50, 30, 20, 99]}), [('Ben', 50), ('Ana', 30), ('Ben', 20)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build an id → name dict, then loop over orders.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.merge(orders, customers, left_on='cust', right_on='id')` — an inner join keeps only matches.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Build an id → name dict, then loop over orders.

```python
def join_orders(customers, orders):
    names = dict(zip(customers["id"], customers["name"]))
    out = []
    for c, a in zip(orders["cust"], orders["amount"]):
        if c in names:
            out.append((names[c], a))
    return out
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.merge(orders, customers, left_on='cust', right_on='id')` — an inner join keeps only matches.

```python
def join_orders(customers, orders):
    merged = pd.merge(pd.DataFrame(orders), pd.DataFrame(customers), left_on="cust", right_on="id", how="inner", sort=False)
    return list(zip(merged["name"], merged["amount"]))
```

**Cost:** O(n + m) time, O(n + m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build an id → name dict, then loop over orders. That costs O(n + m) time and O(n) space."

"I can do better: `pd.merge(orders, customers, left_on='cust', right_on='id')` — an inner join keeps only matches. That's O(n + m) time and O(n + m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/join-customers-and-orders)

<a id="q9"></a>
### Q9. Customers With Their Spending

🟡 Interview standard · *left join + fillna*

Return {name: total spent} for every customer, including those with no orders (0).

**Examples**

- `customers = {'id': [1, 2, 3], 'name': ['Ana', 'Ben', 'Cy']}, orders = {'cust': [2, 1, 2], 'amount': [50, 30, 20]}` → `{'Ana': 30, 'Ben': 70, 'Cy': 0}`

In [ ]:
def spending(customers, orders):
    """
    Return {name: total spent} for every customer, including those with no
    orders (0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spending, [
    (({'id': [1, 2, 3], 'name': ['Ana', 'Ben', 'Cy']}, {'cust': [2, 1, 2], 'amount': [50, 30, 20]}), {'Ana': 30, 'Ben': 70, 'Cy': 0}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum orders per id, then look up every customer with a default of 0.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group the orders, then a **left** merge keeps every customer; fill the gaps with 0.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Sum orders per id, then look up every customer with a default of 0.

```python
def spending(customers, orders):
    spent = {}
    for c, a in zip(orders["cust"], orders["amount"]):
        spent[c] = spent.get(c, 0) + a
    return {name: spent.get(i, 0) for i, name in zip(customers["id"], customers["name"])}
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Group the orders, then a **left** merge keeps every customer; fill the gaps with 0.

```python
def spending(customers, orders):
    totals = pd.DataFrame(orders).groupby("cust", as_index=False)["amount"].sum()
    merged = pd.DataFrame(customers).merge(totals, left_on="id", right_on="cust", how="left")
    merged["amount"] = merged["amount"].fillna(0).astype(int)
    return dict(zip(merged["name"], merged["amount"]))
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum orders per id, then look up every customer with a default of 0. That costs O(n + m) time and O(n) space."

"I can do better: Group the orders, then a **left** merge keeps every customer; fill the gaps with 0. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/customers-with-their-spending)

<a id="q10"></a>
### Q10. Pivot Table

🟡 Interview standard · *pivot\_table*

Return a region × month table of total amounts (missing combinations = 0), regions sorted, months in Jan, Feb, Mar order.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'Jan': [0, 100, 80], 'Feb': [60, 40, 120], 'Mar': [0, 70, 0]}`

In [ ]:
def pivot(data):
    """
    Return a region × month table of total amounts (missing combinations = 0),
    regions sorted, months in Jan, Feb, Mar order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pivot, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'Jan': [0, 100, 80], 'Feb': [60, 40, 120], 'Mar': [0, 70, 0]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dict totals[region][month], then build columns by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.pivot_table(index='region', columns='month', values='amount', aggfunc='sum', fill_value=0)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested dict totals[region][month], then build columns by hand.

```python
def pivot(data):
    months = ["Jan", "Feb", "Mar"]
    regions = sorted(set(data["region"]))
    totals = {}
    for r, m, a in zip(data["region"], data["month"], data["amount"]):
        totals[(r, m)] = totals.get((r, m), 0) + a
    return {m: [totals.get((r, m), 0) for r in regions] for m in months}
```

**Cost:** O(n + R·M) time, O(R·M) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.pivot_table(index='region', columns='month', values='amount', aggfunc='sum', fill_value=0)`.

```python
def pivot(data):
    df = pd.DataFrame(data)
    table = df.pivot_table(index="region", columns="month", values="amount", aggfunc="sum", fill_value=0)
    return table[["Jan", "Feb", "Mar"]]
```

**Cost:** O(n log n) time, O(R·M) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dict totals[region][month], then build columns by hand. That costs O(n + R·M) time and O(R·M) space."

"I can do better: `df.pivot_table(index='region', columns='month', values='amount', aggfunc='sum', fill_value=0)`. That's O(n log n) time and O(R·M) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/pivot-table)

<a id="q11"></a>
### Q11. Wide to Long (melt)

🟡 Interview standard · *melt*

Turn one column per subject into rows of (name, subject, mark), ordered by subject then name.

**Examples**

- `data = {'name': ['Ana', 'Ben'], 'maths': [90, 60], 'art': [70, 85]}` → `[('Ana', 'maths', 90), ('Ben', 'maths', 60), ('Ana', 'art', 70), ('Ben', 'art', 85)]`

In [ ]:
def to_long(data):
    """
    Turn one column per subject into rows of (name, subject, mark), ordered by
    subject then name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_long, [
    (({'name': ['Ana', 'Ben'], 'maths': [90, 60], 'art': [70, 85]},), [('Ana', 'maths', 90), ('Ben', 'maths', 60), ('Ana', 'art', 70), ('Ben', 'art', 85)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over subject columns, then over rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.melt(id_vars='name', var_name='subject', value_name='mark')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over subject columns, then over rows.

```python
def to_long(data):
    out = []
    for subject in [c for c in data if c != "name"]:
        for name, mark in zip(data["name"], data[subject]):
            out.append((name, subject, mark))
    return out
```

**Cost:** O(n·s) time, O(n·s) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.melt(id_vars='name', var_name='subject', value_name='mark')`.

```python
def to_long(data):
    long = pd.DataFrame(data).melt(id_vars="name", var_name="subject", value_name="mark")
    return list(long.itertuples(index=False, name=None))
```

**Cost:** O(n·s) time, O(n·s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over subject columns, then over rows. That costs O(n·s) time and O(n·s) space."

"I can do better: `df.melt(id_vars='name', var_name='subject', value_name='mark')`. That's O(n·s) time and O(n·s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/wide-to-long-melt)

<a id="q12"></a>
### Q12. Cross Tabulation

🟡 Interview standard · *crosstab*

Count sales for each (region, product) pair; return {product: [counts in sorted region order]}.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'ink': [0, 1, 1], 'pen': [1, 2, 1]}`

In [ ]:
def crosstab(data):
    """
    Count sales for each (region, product) pair; return {product: [counts in
    sorted region order]}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(crosstab, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'ink': [0, 1, 1], 'pen': [1, 2, 1]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count pairs in a dict, then lay them out.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.crosstab(df['region'], df['product'])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count pairs in a dict, then lay them out.

```python
def crosstab(data):
    regions, products = sorted(set(data["region"])), sorted(set(data["product"]))
    counts = {}
    for r, p in zip(data["region"], data["product"]):
        counts[(r, p)] = counts.get((r, p), 0) + 1
    return {p: [counts.get((r, p), 0) for r in regions] for p in products}
```

**Cost:** O(n) time, O(R·P) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.crosstab(df['region'], df['product'])`.

```python
def crosstab(data):
    df = pd.DataFrame(data)
    return pd.crosstab(df["region"], df["product"])
```

**Cost:** O(n) time, O(R·P) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count pairs in a dict, then lay them out. That costs O(n) time and O(R·P) space."

"I can do better: `pd.crosstab(df['region'], df['product'])`. That's O(n) time and O(R·P) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/cross-tabulation)

<a id="q13"></a>
### Q13. Share of Group Total

🟡 Interview standard · *transform*

For each sale, return its percentage of its region's total (rounded to 1 decimal).

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `[47.6, 40.0, 19.0, 100.0, 60.0, 33.3]`

In [ ]:
def shares(data):
    """
    For each sale, return its percentage of its region's total (rounded to 1
    decimal).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shares, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), [47.6, 40.0, 19.0, 100.0, 60.0, 33.3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute region totals in one pass, then divide in a second pass.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groupby(...).transform('sum')` returns the group total aligned to every row.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compute region totals in one pass, then divide in a second pass.

```python
def shares(data):
    totals = {}
    for r, a in zip(data["region"], data["amount"]):
        totals[r] = totals.get(r, 0) + a
    return [round(a / totals[r] * 100, 1) for r, a in zip(data["region"], data["amount"])]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `groupby(...).transform('sum')` returns the group total aligned to every row.

```python
def shares(data):
    df = pd.DataFrame(data)
    group_total = df.groupby("region")["amount"].transform("sum")
    return (df["amount"] / group_total * 100).round(1).tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute region totals in one pass, then divide in a second pass. That costs O(n) time and O(k) space."

"I can do better: `groupby(...).transform('sum')` returns the group total aligned to every row. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/share-of-group-total)

<a id="q14"></a>
### Q14. Rank Within Group

🟡 Interview standard · *groupby + rank*

Rank each sale within its region by amount (1 = largest).

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `[1, 2, 3, 1, 1, 2]`

In [ ]:
def ranks(data):
    """
    Rank each sale within its region by amount (1 = largest).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ranks, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), [1, 2, 3, 1, 1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each row, count bigger amounts in the same region.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('region')['amount'].rank(ascending=False, method='min')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** For each row, count bigger amounts in the same region.

```python
def ranks(data):
    out = []
    for r, a in zip(data["region"], data["amount"]):
        bigger = sum(1 for r2, a2 in zip(data["region"], data["amount"]) if r2 == r and a2 > a)
        out.append(bigger + 1)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('region')['amount'].rank(ascending=False, method='min')`.

```python
def ranks(data):
    df = pd.DataFrame(data)
    return df.groupby("region")["amount"].rank(ascending=False, method="min").astype(int).tolist()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each row, count bigger amounts in the same region. That costs O(n²) time and O(n) space."

"I can do better: `df.groupby('region')['amount'].rank(ascending=False, method='min')`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/rank-within-group)

<a id="q15"></a>
### Q15. Running Total per Group

🟡 Interview standard · *groupby + cumsum*

Return each region's running total in row order.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `[100, 80, 140, 60, 200, 210]`

In [ ]:
def running_by_region(data):
    """
    Return each region's running total in row order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_by_region, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), [100, 80, 140, 60, 200, 210]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a running total per region in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('region')['amount'].cumsum()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep a running total per region in a dict.

```python
def running_by_region(data):
    running, out = {}, []
    for r, a in zip(data["region"], data["amount"]):
        running[r] = running.get(r, 0) + a
        out.append(running[r])
    return out
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('region')['amount'].cumsum()`.

```python
def running_by_region(data):
    return pd.DataFrame(data).groupby("region")["amount"].cumsum().tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a running total per region in a dict. That costs O(n) time and O(k) space."

"I can do better: `df.groupby('region')['amount'].cumsum()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/running-total-per-group)

<a id="q16"></a>
### Q16. Monthly Totals From Daily Data

🟡 Interview standard · *Group by month*

Given daily dates and amounts, return totals per month as {'YYYY-MM': total}.

**Examples**

- `data = {'date': ['2024-01-05', '2024-01-20', '2024-02-02', '2024-03-15', '2024-03-16'], 'amount': [10, 15, 7, 3, 4]}` → `{'2024-01': 25, '2024-02': 7, '2024-03': 7}`

In [ ]:
def monthly(data):
    """
    Given daily dates and amounts, return totals per month as {'YYYY-MM':
    total}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monthly, [
    (({'date': ['2024-01-05', '2024-01-20', '2024-02-02', '2024-03-15', '2024-03-16'], 'amount': [10, 15, 7, 3, 4]},), {'2024-01': 25, '2024-02': 7, '2024-03': 7}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice the 'YYYY-MM' part of each date string and sum in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Parse dates, convert to monthly periods and group: `groupby(d.dt.to_period('M'))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Slice the 'YYYY-MM' part of each date string and sum in a dict.

```python
def monthly(data):
    totals = {}
    for d, a in zip(data["date"], data["amount"]):
        totals[d[:7]] = totals.get(d[:7], 0) + a
    return totals
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Parse dates, convert to monthly periods and group: `groupby(d.dt.to_period('M'))`.

```python
def monthly(data):
    df = pd.DataFrame(data)
    month = pd.to_datetime(df["date"]).dt.to_period("M").astype(str)
    return df.groupby(month)["amount"].sum()
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice the 'YYYY-MM' part of each date string and sum in a dict. That costs O(n) time and O(k) space."

"I can do better: Parse dates, convert to monthly periods and group: `groupby(d.dt.to_period('M'))`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/monthly-totals-from-daily-data)

<a id="q17"></a>
### Q17. Rolling 3-Day Average

🟡 Interview standard · *rolling*

Return the 3-day rolling average (the first two days are None), rounded to 1 decimal.

**Examples**

- `data = {'sales': [10, 20, 30, 40, 50]}` → `[None, None, 20.0, 30.0, 40.0]`

In [ ]:
def rolling3(data):
    """
    Return the 3-day rolling average (the first two days are None), rounded to 1
    decimal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rolling3, [
    (({'sales': [10, 20, 30, 40, 50]},), [None, None, 20.0, 30.0, 40.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Average the last 3 values for each position from index 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.rolling(3).mean()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Average the last 3 values for each position from index 2.

```python
def rolling3(data):
    s = data["sales"]
    out = []
    for i in range(len(s)):
        out.append(None if i < 2 else round(sum(s[i - 2:i + 1]) / 3, 1))
    return out
```

**Cost:** O(n·k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.rolling(3).mean()`.

```python
def rolling3(data):
    return pd.DataFrame(data)["sales"].rolling(3).mean().round(1).tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Average the last 3 values for each position from index 2. That costs O(n·k) time and O(n) space."

"I can do better: `s.rolling(3).mean()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/rolling-3-day-average)

<a id="q18"></a>
### Q18. Groups From Two Keys to a Table

🟡 Interview standard · *unstack*

Group by region and product, sum the amounts, and spread products into columns (missing = 0).

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}` → `{'ink': [0, 40, 120], 'pen': [60, 170, 80]}`

In [ ]:
def region_product(data):
    """
    Group by region and product, sum the amounts, and spread products into
    columns (missing = 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(region_product, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]},), {'ink': [0, 40, 120], 'pen': [60, 170, 80]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dict of totals laid out by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groupby(['region', 'product']).sum().unstack(fill_value=0)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested dict of totals laid out by hand.

```python
def region_product(data):
    regions, products = sorted(set(data["region"])), sorted(set(data["product"]))
    totals = {}
    for r, p, a in zip(data["region"], data["product"], data["amount"]):
        totals[(r, p)] = totals.get((r, p), 0) + a
    return {p: [totals.get((r, p), 0) for r in regions] for p in products}
```

**Cost:** O(n) time, O(R·P) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `groupby(['region', 'product']).sum().unstack(fill_value=0)`.

```python
def region_product(data):
    df = pd.DataFrame(data)
    return df.groupby(["region", "product"])["amount"].sum().unstack(fill_value=0)
```

**Cost:** O(n log n) time, O(R·P) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dict of totals laid out by hand. That costs O(n) time and O(R·P) space."

"I can do better: `groupby(['region', 'product']).sum().unstack(fill_value=0)`. That's O(n log n) time and O(R·P) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/groups-from-two-keys-to-a-table)

<a id="q19"></a>
### Q19. Keep Big Groups Only

🟡 Interview standard · *groupby + filter*

Return the amounts from regions that have at least `k` sales.

**Examples**

- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}, k = 2` → `[100, 80, 40, 120, 70]`
- `data = {'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}, k = 3` → `[100, 40, 70]`

In [ ]:
def big_groups(data, k):
    """
    Return the amounts from regions that have at least k sales.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_groups, [
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}, 2), [100, 80, 40, 120, 70]),
    (({'region': ['North', 'South', 'North', 'East', 'South', 'North'], 'product': ['pen', 'pen', 'ink', 'pen', 'ink', 'pen'], 'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar'], 'amount': [100, 80, 40, 60, 120, 70]}, 3), [100, 40, 70]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count per region first, then filter the rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.groupby('region').filter(lambda g: len(g) >= k)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count per region first, then filter the rows.

```python
def big_groups(data, k):
    counts = {}
    for r in data["region"]:
        counts[r] = counts.get(r, 0) + 1
    return [a for r, a in zip(data["region"], data["amount"]) if counts[r] >= k]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.groupby('region').filter(lambda g: len(g) >= k)`.

```python
def big_groups(data, k):
    df = pd.DataFrame(data)
    return df.groupby("region").filter(lambda g: len(g) >= k)["amount"].tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count per region first, then filter the rows. That costs O(n) time and O(k) space."

"I can do better: `df.groupby('region').filter(lambda g: len(g) >= k)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/keep-big-groups-only)

<a id="q20"></a>
### Q20. Customers Without Orders

🟡 Interview standard · *Anti-join*

Return the names of customers who never ordered.

**Examples**

- `customers = {'id': [1, 2, 3, 4], 'name': ['Ana', 'Ben', 'Cy', 'Dia']}, orders = {'cust': [2, 2, 4]}` → `['Ana', 'Cy']`

In [ ]:
def no_orders(customers, orders):
    """
    Return the names of customers who never ordered.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(no_orders, [
    (({'id': [1, 2, 3, 4], 'name': ['Ana', 'Ben', 'Cy', 'Dia']}, {'cust': [2, 2, 4]}), ['Ana', 'Cy']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put ordering ids in a set and keep the others.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Filter with `~customers['id'].isin(orders['cust'])` (or a left merge with `indicator=True`).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Put ordering ids in a set and keep the others.

```python
def no_orders(customers, orders):
    ordered = set(orders["cust"])
    return [n for i, n in zip(customers["id"], customers["name"]) if i not in ordered]
```

**Cost:** O(n + m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Filter with `~customers['id'].isin(orders['cust'])` (or a left merge with `indicator=True`).

```python
def no_orders(customers, orders):
    c = pd.DataFrame(customers)
    return c.loc[~c["id"].isin(orders["cust"]), "name"].tolist()
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put ordering ids in a set and keep the others. That costs O(n + m) time and O(m) space."

"I can do better: Filter with `~customers['id'].isin(orders['cust'])` (or a left merge with `indicator=True`). That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-grouping/customers-without-orders)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does groupby return?</b></summary>

A GroupBy object — a lazy description of the groups. Calling an aggregation (sum, mean, agg), transformation (transform, cumsum) or filter on it produces the result.

</details>

<details>
<summary><b>2. agg vs transform?</b></summary>

agg reduces each group to one row; transform returns a result for every original row (aligned to the original index), so it can be assigned as a new column.

</details>

<details>
<summary><b>3. count vs size?</b></summary>

size counts all rows per group; count counts non-missing values per column.

</details>

<details>
<summary><b>4. merge vs join vs concat?</b></summary>

merge matches rows on key columns (SQL-style); join is a convenience for merging on the index; concat stacks objects along an axis without matching keys.

</details>

<details>
<summary><b>5. pivot vs pivot\_table?</b></summary>

pivot only reshapes and fails if an index/column pair occurs twice; pivot\_table aggregates duplicates with aggfunc and can fill missing cells.

</details>

<details>
<summary><b>6. What is an anti-join and how do you do it in pandas?</b></summary>

Rows in one table with no match in another. Use `~a['key'].isin(b['key'])`, or a left merge with `indicator=True` and keep `_merge == 'left_only'`.

</details>

<details>
<summary><b>7. Why do I get more rows after a merge?</b></summary>

Duplicate keys on both sides create every combination (many-to-many). Deduplicate first or pass `validate=` to catch it.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Split, apply, combine
- [ ] I can explain: groupby by hand
- [ ] I can explain: agg, transform and filter
- [ ] I can explain: Joining tables: merge and concat
- [ ] I can explain: agg vs transform: same size or smaller?
- [ ] I can explain: Joins by hand
- [ ] I can explain: Reshaping: pivot, melt, crosstab, stack
- [ ] I can explain: Time-based grouping and rolling windows
- [ ] I can explain: Wide and long, by hand
- [ ] I can explain: Rolling windows and changes over time, by hand

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Monthly Totals From Daily Data
- [ ] [Q17](#q17) Rolling 3-Day Average
- [ ] [Q18](#q18) Groups From Two Keys to a Table
- [ ] [Q19](#q19) Keep Big Groups Only
- [ ] [Q20](#q20) Customers Without Orders

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.